# Solution B3 · KNNImputer: measuring imputation when you know the true values

**Type**: Implement with scikit-learn

## Objective

Evaluate imputations (mean, median, `KNNImputer` with various `k` and weights) with a clean method: **erase known values** and measure the reconstruction error. Verify that without scaling the `KNNImputer` works worse and that a better imputation does not
automatically translate into a better classification.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_wine
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_wine(return_X_y=True)
mu, sd = X.mean(0), X.std(0)
Z = (X - mu) / sd                                            # standardized truth
mask = np.random.default_rng(0).random(Z.shape) < 0.15       # 15% of the cells missing at random
Zm = np.where(mask, np.nan, Z)
Xm = np.where(mask, np.nan, X)
print(f"{mask.mean():.1%} of cells masked, {mask.sum()} values to reconstruct")

## Commented solution

In [ ]:
def imputation_rmse(imputer):
    R = imputer.fit_transform(Zm)
    return float(np.sqrt(((R[mask] - Z[mask]) ** 2).mean()))          # only the cells we know and have hidden

In [ ]:
base_rmse = {"mean": imputation_rmse(SimpleImputer()), "median": imputation_rmse(SimpleImputer(strategy="median"))}
knn_rmse = pd.Series({k: imputation_rmse(KNNImputer(n_neighbors=k)) for k in [1, 2, 3, 5, 10, 15, 30, 50]})
best_k = int(knn_rmse.idxmin())
knn5_distance = imputation_rmse(KNNImputer(n_neighbors=5, weights="distance"))
print(base_rmse, "\n", knn_rmse.round(3).to_string(), f"\nbest k: {best_k} | k=5 with 1/distance weights: {knn5_distance:.3f}")

In [ ]:
R = KNNImputer(n_neighbors=5).fit_transform(Xm)
Rz = (R - mu) / sd                                                   # I bring the result back to standardized units to compare it with Z
rmse_without_scaling = float(np.sqrt(((Rz[mask] - Z[mask]) ** 2).mean()))
cv = StratifiedKFold(5, shuffle=True, random_state=0)
mean_acc = cross_val_score(make_pipeline(StandardScaler(), SimpleImputer(), LogisticRegression(max_iter=1000)), Xm, y, cv=cv).mean()
knn_acc = cross_val_score(make_pipeline(StandardScaler(), KNNImputer(n_neighbors=5), LogisticRegression(max_iter=1000)), Xm, y, cv=cv).mean()
print(f"RMSE without scaling: {rmse_without_scaling:.3f} (with scaling, k=5: {knn_rmse[5]:.3f})")
print(f"cross-validated accuracy: mean {mean_acc:.3f} | KNNImputer {knn_acc:.3f}")

## Check

In [ ]:
with verify():
    check(close(imputation_rmse(SimpleImputer()), base_rmse["mean"]) and 0.95 < base_rmse["mean"] < 1.15, "RMSE of the mean imputation: about 1 (the standard deviation of the data)", "on the masked cells only")
    check(list(knn_rmse.index) == [1, 2, 3, 5, 10, 15, 30, 50] and best_k == knn_rmse.idxmin(), "RMSE curve as a function of k and best k", "one imputer for every value of k")
    check(knn_rmse.min() < 0.85 * base_rmse["mean"] and knn_rmse[1] > knn_rmse.min() + 0.03, "the KNNImputer reduces the error by over 15% compared to the mean; with k = 1 it is much noisier", "the wine columns are correlated")
    check(3 <= best_k <= 30 and close(knn5_distance, imputation_rmse(KNNImputer(n_neighbors=5, weights="distance"))), "best k between 3 and 30; result with weights computed", "weights='distance'")
    check(rmse_without_scaling > knn_rmse[5] + 0.03, "without scaling the KNNImputer reconstructs worse (over 0.03 more RMSE)", "proline dominates the distance")
    check(abs(knn_acc - mean_acc) < 0.05 and knn_acc > 0.9, "the classification changes little (less than 5 points): the task is easy and the logistic is robust to missing values", "compare knn_acc and mean_acc")
done()

## What you learned

- **The known-truth method:** hide cells you know, impute, compare. It is the only way to measure an imputer without relying on an indirect metric. With the RMSE on the masked cells the mean is worth about 1 (the standard deviation: it uses no
  information from the other columns), the KNNImputer drops to about 0.77: **a quarter less error** because the chemical measurements of the wines are correlated with each other.
- `k` has a U-shaped curve: with `k = 1` the imputation copies a single neighbor (noisy), with a huge `k` it tends to the mean. The maximum is flat between 5 and 30.
- **Scaling is mandatory** here too (the imputer uses distances): without scaling `proline` dominates and the error goes back almost to the level of the mean.
- Improving the imputation does not necessarily improve the downstream model: with an easy problem (wines, logistic) the classification changes by a few points. Whether the imputation matters is measured on the task, not just on the reconstruction.

## Common pitfall

Comparing imputers only on the final accuracy of an easy classifier: differences of one or two points are noise. Also measure the reconstruction error with hidden values, and remember that the imputer must be learned inside the pipeline on the training set alone.

## Interview question

<details><summary>How do you evaluate an imputation method?</summary>

With the known-truth trick: I hide a share of values I know (with the missingness mechanism I consider realistic), impute and measure the error on the hidden cells, besides the effect on the final task in cross-validation with the imputer inside the pipeline.
I remember that the mechanism matters: if the data are missing in a non-random way, the results on random missingness are optimistic.

</details>